# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [5]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [6]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [7]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddon

In [11]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'resume', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'external company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 16 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'source code', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'API endpoints', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'support', 'url': 'https://huggingface.co/support'},
  {'type': 'brand assets'

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/embeddinggemma-2
Updated
8 days ago
•
135k
•
2.23k
jialinyyzz/humanizer
Updated
6 days ago
•
65.6k
•
1.61k
Qwen/Qwen-Image-2.1-Turbo
Updated
2 days ago
•
12.7k
•
765
abenzerps/Qwen-Image-2.1-Uncensored-Quantized
Updated
3 days ago
•
2.32M
•
4.29k
openbmb/MiniCPM-V-4.7-35B-A3B
U

In [19]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/embeddinggemma-2\nUpdated\n8 days ago\n•\n135k\n•\n2.23k\njialinyyzz/humanizer\nUpdated\n6 days ago\n•\n65.6k

In [22]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [23]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is an AI community and platform dedicated to building the future of machine learning. It serves as a collaborative ecosystem where the global AI and machine learning community can create, discover, share, and improve models, datasets, and applications.

The platform hosts over 2 million machine learning models, more than 500,000 datasets, and 1 million+ applications, supporting innovation and democratization of AI technology worldwide.

---

## What We Offer

- **Models**: Access to 2M+ public machine learning models for a variety of tasks including natural language processing, computer vision, and more.
- **Datasets**: Browse and contribute to over 500,000 datasets tailored for machine learning research and applications.
- **Spaces**: A unique platform feature enabling hosting and collaboration of AI apps and demos in an interactive environment.
- **Buckets**: Storage solutions for scalable data and model hosting.
- **HuggingChat**: AI-driven chat interfaces showcasing conversational AI capabilities.
- **Enterprise Solutions**: Comprehensive support and scalable infrastructure for businesses, including inference endpoints, storage, and providers tailored for enterprise-grade deployment.

---

## Our Community & Culture

Hugging Face prides itself on being an open, inclusive, and innovative AI community. The culture emphasizes:

- **Collaboration**: A platform designed to foster teamwork among researchers, developers, and enterprises.
- **Openness**: Encouraging open sharing of knowledge, models, and datasets.
- **Innovation**: Continuously pushing the boundaries of machine learning applications.
- **Learning**: Providing extensive resources such as documentation, blogs, daily research papers, forums, Discord channels, and GitHub repositories to empower the global AI community.

---

## Customers & Partners

Our platform serves a wide range of users including:

- Individual AI researchers and developers
- Academic institutions
- Tech startups and scaleups
- Large enterprises deploying AI at scale
- AI organizations contributing to open-source projects
- Innovators building cutting-edge AI applications

Enterprise customers benefit from dedicated support, custom plans, robust deployment tools, and scalable infrastructure to accelerate AI initiatives.

---

## Careers at Hugging Face

Join a passionate team driving the future of AI! Hugging Face offers exciting career opportunities in areas such as:

- Machine Learning Research and Engineering
- Software Development
- Data Science and Analytics
- Community Management and Developer Advocacy
- Enterprise Support and Sales

We value creativity, collaboration, and a growth mindset. Become part of a mission-driven company empowering the AI community worldwide.

---

## Get Started with Hugging Face

- Explore AI apps and models at [huggingface.co](https://huggingface.co)
- Join the community on Discord and forums
- Access comprehensive documentation and tutorials to boost your ML projects
- Connect through GitHub for open-source contributions
- For business needs, discover enterprise plans and custom solutions

---

**Hugging Face — The AI community building the future.**  
Your hub for state-of-the-art machine learning collaboration and innovation.

---

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face Company Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future of machine learning. It is a leading collaboration platform where machine learning researchers, developers, and enterprises come together to create, discover, and share models, datasets, and applications. With over 2 million models and more than 500,000 datasets available, Hugging Face is the home of machine learning innovation and open collaboration.

---

## What We Offer

- **Models:** Access and contribute to a vast repository of up-to-date machine learning models catering to various AI tasks.
- **Datasets:** Explore and share an extensive collection of datasets spanning numerous fields and use cases.
- **Spaces:** Host and deploy ML applications and demos easily within the community.
- **Buckets:** Scalable storage solutions built for managing data and model assets.
- **Enterprise Solutions:** Tailored services including Hugging Face PRO, Enterprise Support, Inference Providers and Endpoints to meet business-grade needs.
- **Community & Learning:** Engage with a vibrant global community through forums, Discord, GitHub repositories, blog posts, and daily papers.
  
---

## Company Culture

Hugging Face fosters a collaborative, open, and innovation-driven culture. It centers around community engagement, transparency, and the democratization of AI. The company encourages creating and hosting unlimited public machine learning assets, enabling contributors from around the world to collaborate freely and push the boundaries of AI research and deployment.

---

## Our Customers and Community

Hugging Face serves a broad range of users:

- **Researchers and Developers:** Empowering AI practitioners to share and build upon each other’s work.
- **Enterprises and Teams:** Providing robust infrastructure and support for deploying scalable AI solutions.
- **AI Enthusiasts and Learners:** Offering rich educational content and community-driven learning opportunities.
- The platform supports over millions of active users who regularly engage by uploading, downloading, and interacting with cutting-edge AI models and datasets.

---

## Career Opportunities

Hugging Face is continually growing and offers exciting career paths for professionals passionate about AI and open-source collaboration. Joining Hugging Face means becoming part of a mission-driven team committed to building the future of machine learning with a strong community-first mindset.

Careers include roles in:

- Research & Development
- Engineering & Infrastructure
- Community Management & Developer Relations
- Enterprise Sales & Support
- Product Management

---

## Join The Future of AI

Whether you are a developer, researcher, enterprise, or AI enthusiast, Hugging Face invites you to join its platform and community to innovate, collaborate, and build the future of artificial intelligence together.

Explore more and get involved:

- Website: https://huggingface.co
- Community: Discord, Forum, GitHub
- Browse AI Models: Over 2 million available
- Browse Datasets: Over 500,000 datasets

---

**Hugging Face — The AI community building the future.**

In [26]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the premier collaborative platform for the machine learning (ML) community, pioneering the future of artificial intelligence. Serving as a central hub, Hugging Face empowers ML researchers, engineers, scientists, and enthusiasts worldwide to create, share, explore, and experiment with millions of open-source models, datasets, and applications.

Known as *The AI community building the future*, Hugging Face fosters an open, ethical AI environment where collaboration drives innovation and accessibility. It is home to some of the most popular open-source ML libraries and tools, supported by a rapidly growing community and a talented science team pushing the edge of technology.

---

## Platform and Solutions

- **Model Hub:** Access and contribute to over 2 million machine learning models covering a wide variety of tasks.
- **Datasets:** Discover and share from more than 500,000 datasets to power your AI projects.
- **Spaces:** Host interactive ML applications and demos in an easy-to-use environment.
- **Enterprise Solutions:** Team and Enterprise plans include Hugging Face PRO, enterprise support, inference endpoints, storage buckets, and more — designed to accelerate AI deployment at scale with professional-grade performance and reliability.
- **HuggingChat:** AI-powered chat solutions for enhanced user interactions.
- **Inference Providers and Endpoints:** Seamless integration options for machine learning inference in production environments.

The platform supports collaboration by enabling unlimited public hosting, making it easy for users to build and showcase their work while learning from others.

---

## Company Culture

Hugging Face is driven by a passionate and inclusive community ethos. The company values:

- **Collaboration:** A community-first approach encourages sharing knowledge and co-creating innovative AI tools and resources.
- **Openness & Ethics:** Open-source principles guide development, ensuring transparency and responsibility in AI progress.
- **Learning & Growth:** Hugging Face champions education and empowers a new generation of ML professionals by providing accessible resources and a supportive environment.
- **Innovation:** Strong commitment to pushing boundaries in AI research and technology with a dedicated science team exploring cutting-edge advancements.

---

## Customers and Community

Hugging Face supports a diverse range of users including:

- AI researchers and data scientists working in academia and industry.
- Machine learning engineers developing production-ready AI solutions.
- Organizations and enterprises looking to integrate AI with team-focused plans and enterprise-grade services.
- Educators and students learning AI and collaborating on projects.
- Developers building innovative AI applications accessible to the broader public.

The platform thrives on an active, vibrant community that connects globally via forums, Discord channels, GitHub repositories, and more.

---

## Careers at Hugging Face

Join a fast-growing AI leader shaping the future of open-source machine learning. Hugging Face offers roles in:

- Research and Data Science
- Software Engineering and Development
- Community Management and Support
- Enterprise Solutions and Customer Success
- Product and Design

Working at Hugging Face means being part of a mission-driven company focused on innovation, ethical AI, and global collaboration. Employees benefit from an inclusive environment fostering creativity, continuous learning, and the opportunity to impact millions of users worldwide.

---

## Get Involved

- Explore models, datasets, and more at [huggingface.co](https://huggingface.co)
- Join the community on [Discord](https://discord.gg/huggingface) and [GitHub](https://github.com/huggingface)
- Discover enterprise solutions and ask about customized team plans
- Check current job openings and apply to be part of the future of AI

---

**Hugging Face** — Empowering AI practitioners to build an open, cooperative, and pioneering AI future together.

---

*Contact & Social*  
Website: https://huggingface.co  
Twitter: @huggingface | LinkedIn: hugging-face | GitHub: huggingface

#BuildTheFuture #OpenAI #MachineLearning #AIForEveryone

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>